# Export MEX for 10x Panel Design Tool

Important stuff:
* Needs to be raw counts to work
* If not raw counts they will email you and ask you to transfer it to them through their annoying command line tool
* Matrix Exchange (MEX) format: A .zip or .tar file containing four files (matrix.mtx.gz, barcodes.tsv.gz, features.tsv.gz, and a .csv or .tsv of cell type annotations).

In [1]:
from pathlib import Path
import sys
import os
import pandas as pd
import scipy.io
import scipy.sparse
import scanpy as sc
import tempfile
import gzip
import zipfile
import session_info

In [2]:
from pathlib import Path

base_dir = Path('/home/workspace/private/projects/kim/drg')

panel_dir = base_dir / 'panel/gene-lists/output'
adata_path = base_dir / 'data/scrna-seq/h5ad/04_clustered/GSE139088-scvi-leiden.h5ad'
bdata_path = base_dir / 'data/scrna-seq/h5ad/04_clustered/GSE254789-nonneurons.h5ad'
output_dir = base_dir / 'data/scrna-seq/mex/2025-07-09/'

output_dir.mkdir(parents = True, exist_ok = True)

In [3]:
adata = sc.read_h5ad(adata_path)
bdata = sc.read_h5ad(bdata_path)

In [4]:
print(adata.layers)
print(bdata.layers)

Layers with keys: counts, log1p, normalized_1e6
Layers with keys: counts, log1p, normalized_1e6


In [5]:
adata.X = adata.layers['counts'].copy() # 10x wants raw counts
bdata.X = bdata.layers['counts'].copy()

In [6]:
adata.X = adata.X.astype('int32') #10x wants like this
bdata.X = bdata.X.astype('int32') 

In [7]:
# confirm raw counts bc paranoia
print(adata.X[:10,:10])
print('-'*20)
print(bdata.X[:10,:10].toarray())

[[0 4 0 0 2 0 1 5 0 0]
 [2 2 1 1 2 0 0 0 0 0]
 [0 2 1 0 0 0 1 0 0 2]
 [3 1 0 0 2 2 2 1 0 0]
 [2 3 0 0 4 1 2 7 0 0]
 [2 3 0 1 2 1 1 4 0 0]
 [0 1 0 0 1 0 0 3 0 0]
 [0 0 0 0 0 0 1 0 0 0]
 [1 0 0 1 0 0 2 2 0 1]
 [1 1 0 0 1 0 1 4 0 0]]
--------------------
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 5 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 4 0 0 0 0]]


In [8]:
# Gene name update mapping
gene_updates = {
    "Fam19a1": "Tafa1",
    "Mrgprc11": "Mrgprx1",
    "Cyr61": "Ccn1",
    "2810417H13Rik": "Pclaf",
    "Fam19a5": "Tafa5",
    "Fam19a4": "Tafa4",
    "Abcb1": "Abcb1a"
}

adata.var_names = adata.var_names.to_series().replace(gene_updates)
bdata.var_names = bdata.var_names.to_series().replace(gene_updates)

In [9]:
found_adata = [new for new in gene_updates.values() if new in adata.var_names]
print(f"Found in {adata.uns.get('sample_name', 'adata')}:", found_adata)

found_bdata = [new for new in gene_updates.values() if new in bdata.var_names]
print(f"Found in {bdata.uns.get('sample_name', 'bdata')}:", found_bdata)

Found in adata: ['Tafa1', 'Mrgprx1', 'Ccn1', 'Pclaf', 'Tafa5', 'Tafa4', 'Abcb1a']
Found in bdata: ['Tafa1', 'Mrgprx1', 'Ccn1', 'Pclaf', 'Tafa5', 'Tafa4', 'Abcb1a']


In [10]:
adata.obs

,original_annotation,GSE,GSM,sample_number,sample_id,experiment_id,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_ribosomal,pct_counts_ribosomal,doublet_scores,predicted_doublets,_scvi_batch,_scvi_labels,leiden
GSM4130750_AAACCTGTCTGCTGCT-1,CGRP-Theta,GSE139088,GSM4130750,1,GSE139088_1,GSE139088,2523,6931.0,134.0,1.933343,326.0,4.703506,0.070248,False,0,0,5
GSM4130750_AAACGGGAGTACGCGA-1,CGRP-Gamma,GSE139088,GSM4130750,1,GSE139088_1,GSE139088,5212,19993.0,430.0,2.150753,1430.0,7.152503,0.044776,False,0,0,9
GSM4130750_AAACGGGCAGGGATTG-1,CGRP-Alpha,GSE139088,GSM4130750,1,GSE139088_1,GSE139088,4349,13737.0,576.0,4.193055,1143.0,8.320594,0.025710,False,0,0,0
GSM4130750_AAACGGGCATTGCGGC-1,CGRP-Alpha,GSE139088,GSM4130750,1,GSE139088_1,GSE139088,4747,18238.0,490.0,2.686698,1593.0,8.734510,0.036697,False,0,0,0
GSM4130750_AAACGGGGTGGGTATG-1,Nonpeptidergic nociceptors,GSE139088,GSM4130750,1,GSE139088_1,GSE139088,4670,20604.0,514.0,2.494661,1450.0,7.037468,0.036697,False,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM4130750_CCACTACAGATGCCAG-6,Abeta-Field,GSE139088,GSM4130750,6,GSE139088_6,GSE139088,3008,7677.0,168.0,2.188355,381.0,4.962876,0.106870,False,5,0,8
GSM4130750_CCAGCGAAGCTAGCCC-6,CGRP-Alpha,GSE139088,GSM4130750,6,GSE139088_6,GSE139088,3811,10125.0,412.0,4.069136,948.0,9.362963,0.034335,False,5,0,0
GSM4130750_CCAGCGAAGTGGAGTC-6,Nonpeptidergic nociceptors,GSE139088,GSM4130750,6,GSE139088_6,GSE139088,4833,21559.0,692.0,3.209796,2056.0,9.536620,0.041667,False,5,0,1
GSM4130750_CCAGCGACAAGAGTCG-6,C-LTMR,GSE139088,GSM4130750,6,GSE139088_6,GSE139088,4843,17804.0,835.0,4.689957,1185.0,6.655808,0.285714,False,5,0,6


In [11]:
bdata.obs

,GSE,GSM,sample_number,sample_id,experiment_id,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_ribosomal,pct_counts_ribosomal,doublet_scores,predicted_doublets,_scvi_batch,_scvi_labels,leiden,artifact,class,cell_type
GSM8057824_AAACCCACAGCATCTA-1,GSE254789,GSM8057824,1,GSE254789_1,GSE254789,1053,1638.0,26.0,1.587302,107.0,6.532357,0.049965,False,0,0,5,OK,Non-neuron,Schwann_cell
GSM8057824_AAACCCAGTACTAGCT-1,GSE254789,GSM8057824,1,GSE254789_1,GSE254789,868,1470.0,49.0,3.333333,78.0,5.306122,0.055432,False,0,0,3,OK,Non-neuron,SGC
GSM8057824_AAACCCAGTCATCTAG-1,GSE254789,GSM8057824,1,GSE254789_1,GSE254789,1821,3191.0,155.0,4.857411,130.0,4.073958,0.132262,False,0,0,4,OK,Non-neuron,SGC
GSM8057824_AAACCCAGTTGCCGAC-1,GSE254789,GSM8057824,1,GSE254789_1,GSE254789,1108,2456.0,22.0,0.895766,137.0,5.578176,0.027298,False,0,0,3,OK,Non-neuron,SGC
GSM8057824_AAACCCATCTTACCAT-1,GSE254789,GSM8057824,1,GSE254789_1,GSE254789,3017,7703.0,221.0,2.869012,474.0,6.153447,0.066451,False,0,0,3,OK,Non-neuron,SGC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM8057829_TTTGGAGAGTCGGGAT-1,GSE254789,GSM8057829,6,GSE254789_6,GSE254789,922,1288.0,37.0,2.872671,66.0,5.124224,0.067857,False,5,0,2,OK,Non-neuron,SGC
GSM8057829_TTTGGAGTCCAAACCA-1,GSE254789,GSM8057829,6,GSE254789_6,GSE254789,921,1442.0,114.0,7.905687,68.0,4.715672,0.021518,False,5,0,4,OK,Non-neuron,SGC
GSM8057829_TTTGGTTTCCAACCGG-1,GSE254789,GSM8057829,6,GSE254789_6,GSE254789,779,1209.0,48.0,3.970223,64.0,5.293631,0.062340,False,5,0,4,OK,Non-neuron,SGC
GSM8057829_TTTGTTGTCCCTTTGG-1,GSE254789,GSM8057829,6,GSE254789_6,GSE254789,786,1235.0,72.0,5.829959,87.0,7.044534,0.036145,False,5,0,2,OK,Non-neuron,SGC


In [13]:
print("adata.var index name:", adata.var.index.name)
print("adata.var index example:", adata.var.index[:5])
print("adata.var columns:", adata.var.columns)
adata.var.head()

adata.var index name: None
adata.var index example: Index(['0610007P14Rik', '0610009B22Rik', '0610009L18Rik', '0610009O20Rik',
       '0610010F05Rik'],
      dtype='object')
adata.var columns: Index(['mt', 'ribosomal', 'n_cells_by_counts', 'mean_counts',
       'pct_dropout_by_counts', 'total_counts'],
      dtype='object')


,mt,ribosomal,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
0610007P14Rik,False,False,6287,1.125146,43.558668,12533.0
0610009B22Rik,False,False,7134,1.895682,35.954754,21116.0
0610009L18Rik,False,False,2742,0.306491,75.383787,3414.0
0610009O20Rik,False,False,1629,0.162851,85.375707,1814.0
0610010F05Rik,False,False,8274,1.754825,25.720442,19547.0


In [14]:
print("bdata.var index name:", bdata.var.index.name)
print("bdata.var index example:", bdata.var.index[:5])
print("bdata.var columns:", bdata.var.columns)
bdata.var.head()

bdata.var index name: None
bdata.var index example: Index(['Xkr4', 'Gm1992', 'Gm37381', 'Rp1', 'Rp1-1'], dtype='object')
bdata.var columns: Index(['mt', 'ribosomal', 'n_cells_by_counts', 'mean_counts',
       'pct_dropout_by_counts', 'total_counts'],
      dtype='object')


,mt,ribosomal,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
Xkr4,False,False,4179,0.051962,95.501373,4827.0
Gm1992,False,False,179,0.001948,99.807309,181.0
Gm37381,False,False,3,0.000032,99.996771,3.0
Rp1,False,False,73,0.000786,99.921417,73.0
Rp1-1,False,False,0,0.000000,100.000000,0.0


In [12]:
adata.var.index.name = "gene_id" # assign new index name for bdata
adata.var

,mt,ribosomal,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
gene_id,,,,,,
0610007P14Rik,False,False,6287,1.125146,43.558668,12533.0
0610009B22Rik,False,False,7134,1.895682,35.954754,21116.0
0610009L18Rik,False,False,2742,0.306491,75.383787,3414.0
0610009O20Rik,False,False,1629,0.162851,85.375707,1814.0
0610010F05Rik,False,False,8274,1.754825,25.720442,19547.0
...,...,...,...,...,...,...
Zyg11a,False,False,0,0.000000,100.000000,0.0
Zyg11b,False,False,8463,1.715594,24.023701,19110.0
Zyx,False,False,3661,0.455606,67.133495,5075.0


In [13]:
bdata.var.index.name = "gene_id"
bdata.var

,mt,ribosomal,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
gene_id,,,,,,
Xkr4,False,False,4179,0.051962,95.501373,4827.0
Gm1992,False,False,179,0.001948,99.807309,181.0
Gm37381,False,False,3,0.000032,99.996771,3.0
Rp1,False,False,73,0.000786,99.921417,73.0
Rp1-1,False,False,0,0.000000,100.000000,0.0
...,...,...,...,...,...,...
DHRSX,False,False,6013,0.070445,93.527100,6544.0
Vmn2r122,False,False,0,0.000000,100.000000,0.0
CAAA01147332.1,False,False,1464,0.016212,98.424027,1506.0


In [14]:
# adata, neurons
output_zip = output_dir / "GSE139088-matrix.zip"  # Final Xenium-compatible output
gene_id_key = "gene_id"                          # This is the index of adata.var
cell_type_key = "original_annotation"           # Column in adata.obs with cell labels

# Ensure the index of adata.var is named
adata.var.index.name = gene_id_key

# Use the index as both gene_id and gene_name
genes = adata.var.reset_index()[[gene_id_key]]
genes["gene_name"] = genes[gene_id_key]
genes["feature_type"] = ["Gene Expression"] * len(genes)

# Barcodes and cell types
barcodes = pd.DataFrame(adata.obs.index)
celltypes = adata.obs[[cell_type_key]].reset_index()
celltypes.columns = ["barcode", "annotation"]

# Write files into a temp folder and zip
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)

    # matrix.mtx.gz
    with gzip.open(tmp_path / "matrix.mtx.gz", "wb") as f:
        scipy.io.mmwrite(f, scipy.sparse.csc_matrix(adata.X.T))

    # features.tsv.gz
    genes.to_csv(tmp_path / "features.tsv.gz", sep="\t", index=False, header=False, compression="gzip")

    # barcodes.tsv.gz
    barcodes.to_csv(tmp_path / "barcodes.tsv.gz", sep="\t", index=False, header=False, compression="gzip")

    # celltypes.csv
    celltypes.to_csv(tmp_path / "celltypes.csv", index=False)

    # zip all
    with zipfile.ZipFile(output_zip, "w") as zipf:
        for fname in ["matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz", "celltypes.csv"]:
            zipf.write(tmp_path / fname, arcname=fname)

print(output_zip)

/home/workspace/private/projects/kim/drg/data/scrna-seq/mex/2025-07-09/GSE139088-matrix.zip


In [ ]:
# bdata, neurons
output_zip = output_dir / "GSE254789-matrix.zip"  # Final Xenium-compatible output
gene_id_key = "gene_id"
cell_type_key = "cell_type"

# Ensure the index of bdata.var is named
bdata.var.index.name = gene_id_key

# Use the index as both gene_id and gene_name
genes = bdata.var.reset_index()[[gene_id_key]]
genes["gene_name"] = genes[gene_id_key]
genes["feature_type"] = ["Gene Expression"] * len(genes)

# Barcodes and cell types
barcodes = pd.DataFrame(bdata.obs.index)
celltypes = bdata.obs[[cell_type_key]].reset_index()
celltypes.columns = ["barcode", "annotation"]

# Write files into a temp folder and zip
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)

    # matrix.mtx.gz
    with gzip.open(tmp_path / "matrix.mtx.gz", "wb") as f:
        scipy.io.mmwrite(f, scipy.sparse.csc_matrix(bdata.X.T))

    # features.tsv.gz
    genes.to_csv(tmp_path / "features.tsv.gz", sep="\t", index=False, header=False, compression="gzip")

    # barcodes.tsv.gz
    barcodes.to_csv(tmp_path / "barcodes.tsv.gz", sep="\t", index=False, header=False, compression="gzip")

    # celltypes.csv
    celltypes.to_csv(tmp_path / "celltypes.csv", index=False)

    # zip all
    with zipfile.ZipFile(output_zip, "w") as zipf:
        for fname in ["matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz", "celltypes.csv"]:
            zipf.write(tmp_path / fname, arcname=fname)

print(output_zip)